# ML4SCI Genie GSoC - Common Task 2: Jets as Graphs
This notebook implements an end-to-end pipeline to classify quark vs. gluon jets using Graph Neural Networks (GNNs).
It processes the `quark-gluon_data-set_n139306.hdf5` efficiently by strictly operating on chunks to stay under standard RAM limits (16GB).

**Pipeline Overview:**
1. **Phase 1: Sequential Graph Builder (Pre-processing)**
   - Sequentially read the HDF5 file in chunks (matching compression blocks).
   - Convert images to point clouds: Non-zero pixels become nodes with features `[x, y, E_ecal, E_hcal, E_track]`.
   - Build edges using spatial $k$-Nearest Neighbors.
   - Save chunked PyTorch Geometric (PyG) graphs to disk.
2. **Phase 2: PyTorch Geometric Custom Dataset**
   - Create a Dataset that loads one chunk at a time, shuffles locally, and serves mini-batches.
3. **Phase 3: Static GraphSAGE Model Setup**
   - Build a Message Passing GNN compatible with our static $k$-NN graphs.
4. **Phase 4: Training & Evaluation Loop**
   - Train and calculate AUC-ROC on the validation set.


In [ ]:
import os
import h5py
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool
from torch_cluster import knn_graph
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Constants and seeds
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

HDF5_PATH = 'quark-gluon_data-set_n139306.hdf5'
CHUNK_SIZE = 6400  # Matches LZF compression chunks in the dataset
PROCESSED_DIR = 'processed_graphs'
os.makedirs(PROCESSED_DIR, exist_ok=True)


In [ ]:
def convert_image_to_graph(image_tensor, label, k=8):
    '''
    Converts a single (3, 125, 125) jet image into a graph.
    Nodes: Non-zero pixels across any channel.
    Node Features: [x, y, E_ecal, E_hcal, E_track]
    Edges: k-Nearest Neighbors based on spatial (x, y) coordinates.
    '''
    # Flatten spatial dimensions
    # image_tensor shape: (3, 125, 125)
    c, h, w = image_tensor.shape
    
    # Create coordinate grids
    y_coords, x_coords = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    
    # Check if a pixel is non-zero in ANY of the 3 channels
    mask = np.any(image_tensor > 0, axis=0) # shape (125, 125)
    
    if not np.any(mask):
        # Empty image fallback
        return None
        
    # Extract node features
    x_valid = x_coords[mask]
    y_valid = y_coords[mask]
    features_valid = image_tensor[:, mask].T # shape (N_nodes, 3)
    
    # Normalize coordinates to unit interval approximately
    x_norm = x_valid / float(w)
    y_norm = y_valid / float(h)
    
    # Construct final node feature matrix [x, y, ecal, hcal, track]
    x = torch.tensor(np.column_stack([x_norm, y_norm, features_valid]), dtype=torch.float)
    y_label = torch.tensor([label], dtype=torch.long)
    
    # Construct k-NN edges based strictly on spatial coordinates (first 2 features: x, y)
    pos = x[:, :2] 
    
    # Check if number of nodes is less than k
    actual_k = min(k, pos.size(0) - 1)
    if actual_k <= 0:
        return None
        
    edge_index = knn_graph(pos, k=actual_k, loop=False)
    
    return Data(x=x, edge_index=edge_index, y=y_label)

def build_sequential_graphs(filepath, out_dir, chunk_size=CHUNK_SIZE):
    '''
    Sequentially reads the HDF5 file in chunks and saves processed graphs.
    '''
    print("Starting Sequential Graph Conversion...")
    with h5py.File(filepath, 'r') as f:
        N = f['X_jets'].shape[0]
        num_chunks = int(np.ceil(N / chunk_size))
        
        for i in range(num_chunks):
            start_idx = i * chunk_size
            end_idx = min((i + 1) * chunk_size, N)
            
            chunk_file = os.path.join(out_dir, f'chunk_{i}.pt')
            if os.path.exists(chunk_file):
                print(f"Chunk {i+1}/{num_chunks} already exists. Skipping.")
                continue
                
            print(f"Processing Chunk {i+1}/{num_chunks} (Indices {start_idx}:{end_idx})...")
            
            # 1. Read contiguous slice (decompress strictly once)
            X_chunk = f['X_jets'][start_idx:end_idx]
            y_chunk = f['y'][start_idx:end_idx]
            
            # 2. Convert to graphs
            graph_list = []
            for j in range(X_chunk.shape[0]):
                g = convert_image_to_graph(X_chunk[j], y_chunk[j])
                if g is not None:
                    graph_list.append(g)
                    
            # 3. Save chunk to disk
            torch.save(graph_list, chunk_file)
            print(f"Saved {len(graph_list)} graphs to {chunk_file}")
            
build_sequential_graphs(HDF5_PATH, PROCESSED_DIR)


In [ ]:
import glob
import random

class ChunkedGraphDataset(Dataset):
    '''
    A custom PyG Dataset that reads static graph chunks sequentially,
    shuffling the internal graphs to maintain a low memory footprint
    while still providing randomness across epochs.
    '''
    def __init__(self, processed_dir, transform=None, pre_transform=None):
        super().__init__(processed_dir, transform, pre_transform)
        self.chunk_files = sorted(glob.glob(os.path.join(processed_dir, 'chunk_*.pt')))
        
        # We need the total length. We will load each chunk once initially to count, 
        # or we can read a prepopulated list. For a quick implementation, we load each to count.
        self.total_len = 0
        self.chunk_lengths = []
        print("Calculating dataset length...")
        for c in self.chunk_files:
            c_data = torch.load(c)
            self.total_len += len(c_data)
            self.chunk_lengths.append(len(c_data))
            
        print(f"Total Graphs: {self.total_len}")
        
        # Buffer to hold current chunk
        self._current_chunk_idx = -1
        self._current_chunk_data = []

    def len(self):
        return self.total_len

    def get(self, idx):
        # Find which chunk this idx belongs to
        chunk_idx = 0
        local_idx = idx
        while chunk_idx < len(self.chunk_lengths) and local_idx >= self.chunk_lengths[chunk_idx]:
            local_idx -= self.chunk_lengths[chunk_idx]
            chunk_idx += 1
            
        if chunk_idx != self._current_chunk_idx:
            # We must load a new chunk!
            self._current_chunk_data = torch.load(self.chunk_files[chunk_idx])
            self._current_chunk_idx = chunk_idx
            
        return self._current_chunk_data[local_idx]

dataset = ChunkedGraphDataset(PROCESSED_DIR)

# Split dataset into train and validation (80-20 split)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)


In [ ]:
class JetGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(JetGraphSAGE, self).__init__()
        # Input features: [x, y, E_ecal, E_hcal, E_track] (in_channels=5)
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        
        # Global pooling and classification head
        self.lin1 = torch.nn.Linear(hidden_channels, hidden_channels // 2)
        self.lin2 = torch.nn.Linear(hidden_channels // 2, out_channels)

    def forward(self, x, edge_index, batch):
        # 1. Message Passing
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        
        x = self.conv3(x, edge_index)
        x = F.relu(x)
        
        # 2. Aggregation / Global Pooling
        # Average node features across the entire graph to form a single vector per graph
        x = global_mean_pool(x, batch)
        
        # 3. Predict via MLP
        x = self.lin1(x)
        x = F.relu(x)
        x = self.lin2(x)
        
        return x

model = JetGraphSAGE(in_channels=5, hidden_channels=64, out_channels=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = torch.nn.CrossEntropyLoss()
print(model)


In [ ]:
def train():
    model.train()
    total_loss = 0
    
    # We use a progress bar
    pbar = tqdm(train_loader, desc="Training")
    for data in pbar:
        data = data.to(device)
        optimizer.zero_grad()
        
        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * data.num_graphs
        pbar.set_postfix({'loss': loss.item()})
        
    return total_loss / len(train_loader.dataset)

@torch.no_grad()
def test(loader):
    model.eval()
    all_preds = []
    all_targets = []
    
    for data in tqdm(loader, desc="Evaluating"):
        data = data.to(device)
        out = model(data.x, data.edge_index, data.batch)
        probs = F.softmax(out, dim=1)[:, 1]  # Probabilities for class 1 (Gluon)
        
        all_preds.append(probs.cpu().numpy())
        all_targets.append(data.y.cpu().numpy())
        
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    
    # Calculate metrics
    binary_preds = (all_preds > 0.5).astype(int)
    auc = roc_auc_score(all_targets, all_preds)
    acc = accuracy_score(all_targets, binary_preds)
    
    return auc, acc

epochs = 5
for epoch in range(1, epochs + 1):
    loss = train()
    val_auc, val_acc = test(val_loader)
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}, Val Acc: {val_acc:.4f}')
